# Cross-jurisdiction Alignment (LLM-Validated)

**Scope:** Stage 5B. Aligns canonical classes across jurisdictions using a two-stage hybrid: embedding-based candidate generation followed by LLM validation against ODP-11 (Functional Equivalence Pattern) criteria.

---

## Method

```
Step 1: Embedding candidate generation (no LLM)
  - BGE-M3 embed (label + definition) per class
  - Pairwise cosine similarity
  - Filter: same CCO parent, different jurisdiction, similarity >= 0.70
  - Output: ~80-100 candidate pairs (recall-oriented)

Step 2: LLM validation per candidate 
  - Model: Qwen3-32B via Nscale (distinct from Stage 2's DeepSeek-V3.1)
  - Prompt provides: Class A context, Class B context, ODP-11 criteria
  - LLM returns structured JSON: decision + per-criterion pass/fail + justification
  
Step 3: Build alignment artifacts (no LLM)
  - Filter to ALIGN decisions
  - Union-find clustering (transitive equivalence)
  - Emit owl:equivalentClass axioms
```




In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'sentence-transformers', 'rdflib', 'scikit-learn', 'openai'], check=False)
print('Dependencies ready.')

In [ ]:
import json, os, re, time
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from rdflib import Graph, URIRef, Namespace, RDF, OWL, RDFS
from openai import OpenAI
print('Imports loaded.')

In [ ]:
# -- Stage 5B FULL RUN configuration ----------------------------------------
BASE_DIR = Path('/Users/umair/Synthesising Regulatory Ontologies')

# INPUT: Stage 5A full run output
STAGE5A_DIR = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'stage5a_dedup'
CANONICAL_CLASSES_PATH = STAGE5A_DIR / 'stage5a_canonical_classes.json'

# OUTPUT: Stage 5B full run
OUTPUT_DIR = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'stage5b_alignment'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files
ALIGNMENT_MANIFEST = OUTPUT_DIR / 'stage5b_alignment_manifest.json'
ALIGNMENT_TTL      = OUTPUT_DIR / 'cross_jurisdiction_alignments.ttl'
RAW_RESPONSES      = OUTPUT_DIR / 'raw_llm_responses.jsonl'
CHECKPOINT_DIR     = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Embedding model — same as Stage 5A
EMBED_MODEL = 'BAAI/bge-m3'

# Threshold tuned based on pilot analysis (0.65→too noisy, 0.70→sweet spot)
SIM_THRESHOLD = 0.70

# Constraints
SAME_PARENT_REQUIRED = True

# Jurisdictions and namespaces
JURISDICTIONS = ['gro-uk', 'gro-us', 'gro-ca', 'gro-au']
JUR_NAMESPACES = {
    'gro-uk': 'https://w3id.org/cco-gro/onto/uk#',
    'gro-us': 'https://w3id.org/cco-gro/onto/us#',
    'gro-ca': 'https://w3id.org/cco-gro/onto/ca#',
    'gro-au': 'https://w3id.org/cco-gro/onto/au#',
    'gro':    'https://w3id.org/cco-gro/onto#',
    'cco':    'https://www.w3id.org/cco/cco#',
}

# LLM configuration
LLM_MODEL    = 'Qwen/Qwen3-32B'
LLM_BASE_URL = 'https://inference.api.nscale.com/v1'
# ★ SECURITY: Reads API key from environment, NOT hardcoded
LLM_API_KEY = '' # your api key
LLM_TEMPERATURE = 0.0
LLM_MAX_TOKENS  = 1500

# Checkpoint configuration
CHECKPOINT_INTERVAL = 25  # Save every 25 candidates
RESUME_FROM_CHECKPOINT = True  # Auto-resume if checkpoint exists

# Verify configuration
if not LLM_API_KEY:
    raise RuntimeError('NSCALE_API_KEY not set in environment. Run: export NSCALE_API_KEY="your_key"')

assert CANONICAL_CLASSES_PATH.exists(), f'Stage 5A canonical classes not found: {CANONICAL_CLASSES_PATH}'

print(f' Configuration verified')
print(f'Stage 5A input:    {CANONICAL_CLASSES_PATH.name}')
print(f'Output dir:        {OUTPUT_DIR}')
print(f'Embedding model:   {EMBED_MODEL}')
print(f'LLM model:         {LLM_MODEL}')
print(f'Threshold:         {SIM_THRESHOLD}')
print(f'Checkpoint every:  {CHECKPOINT_INTERVAL} candidates')
print(f'Resume enabled:    {RESUME_FROM_CHECKPOINT}')

In [ ]:
# -- Load 5A canonical classes ------------------------------------------------
with open(CANONICAL_CLASSES_PATH, encoding='utf-8') as f:
    canon = json.load(f)

all_classes = []
for jur, classes in canon['per_jurisdiction'].items():
    for c in classes:
        all_classes.append({
            'jurisdiction': jur,
            'class':        c['class'],
            'parent':       c['parent'],
            'label':        c['label'],
            'definition':   c['definition'],
        })

print(f'Total canonical classes (5A output): {len(all_classes)}')
print()
print('Per jurisdiction:')
jc = Counter(c['jurisdiction'] for c in all_classes)
for j in JURISDICTIONS:
    print(f'  {j}: {jc.get(j, 0)} classes')

print('\nTop CCO parents:')
pc = Counter(c['parent'] for c in all_classes)
for p, n in pc.most_common(10):
    jurs = set(c['jurisdiction'] for c in all_classes if c['parent'] == p)
    print(f'  {p:40s}: {n:4d} classes across {len(jurs)} jurisdiction(s)')

In [ ]:
# -- Step 1a: Embed all classes -----------------------------------------------
print(f'Loading {EMBED_MODEL}...')
model = SentenceTransformer(EMBED_MODEL)
print(f'  Embedding dim: {model.get_sentence_embedding_dimension()}')

texts = [f"{c['label']}. {c['definition']}" for c in all_classes]
print(f'Encoding {len(texts)} class descriptions...')
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64,
)
print(f' {len(all_classes)} classes embedded; dim = {embeddings.shape[1]}')

In [ ]:
# -- Step 1b: Generate candidate pairs ---------------------------------------
print(f'Computing pairwise similarities (this may take a moment for {len(all_classes)} classes)...')
sim = cosine_similarity(embeddings)

print(f'Filtering: same parent, different jurisdiction, sim >= {SIM_THRESHOLD}')
candidates = []
for i in range(len(all_classes)):
    for j in range(i+1, len(all_classes)):
        ci, cj = all_classes[i], all_classes[j]
        if ci['jurisdiction'] == cj['jurisdiction']: continue
        if SAME_PARENT_REQUIRED and ci['parent'] != cj['parent']: continue
        s = float(sim[i, j])
        if s >= SIM_THRESHOLD:
            candidates.append({
                'similarity': round(s, 4),
                'i': i, 'j': j,
                'class_a': ci, 'class_b': cj,
            })

candidates.sort(key=lambda x: x['similarity'], reverse=True)

print(f'\n Embedding candidates above {SIM_THRESHOLD}: {len(candidates)}')

# Estimated cost/time
est_minutes = len(candidates) * 2.5 / 60
print(f'  Estimated runtime: ~{est_minutes:.0f} minutes ({len(candidates)} × 2.5 sec/call)')

# Distribution analysis
print('\nDistribution by CCO parent:')
bp = Counter(c['class_a']['parent'] for c in candidates)
for p, n in bp.most_common():
    print(f'  {p}: {n}')

print('\nDistribution by jurisdiction pair:')
bjp = Counter()
for c in candidates:
    pair = tuple(sorted([c['class_a']['jurisdiction'], c['class_b']['jurisdiction']]))
    bjp[pair] += 1
for pair, n in bjp.most_common():
    print(f'  {pair[0]} ↔ {pair[1]}: {n}')

print(f'\nTop 10 candidates by similarity:')
for c in candidates[:10]:
    a, b = c['class_a'], c['class_b']
    print(f'  [{c["similarity"]:.3f}] {a["jurisdiction"]}/{a["class"].split(":")[-1]:35s} ↔ {b["jurisdiction"]}/{b["class"].split(":")[-1]}')

In [ ]:
# -- Step 2 prompt definition: Functional Equivalence with alignment relation -
LLM_SYSTEM = """You are an ontology engineering analyst evaluating cross-jurisdiction class alignment under the Functional Equivalence Pattern.

Your task: decide whether two domain classes from different regulatory jurisdictions are functionally equivalent, and if so, determine the appropriate alignment relation.

You must check each criterion independently and return a structured JSON decision.

Respond with valid JSON only. No prose, no markdown fences."""

LLM_USER_TEMPLATE = """Evaluate whether these two classes are functionally equivalent.

CLASS A:
  Jurisdiction: {jur_a}
  Class URI:    {uri_a}
  CCO parent:   {parent_a}
  Label:        "{label_a}"
  Definition:   "{def_a}"

CLASS B:
  Jurisdiction: {jur_b}
  Class URI:    {uri_b}
  CCO parent:   {parent_b}
  Label:        "{label_b}"
  Definition:   "{def_b}"

Embedding similarity (for context only): {sim}

FUNCTIONAL EQUIVALENCE PATTERN CRITERIA (check each independently):

1. **same_cco_parent**: Both classes share the same CCO parent class.
   (PASS only if both have the exact same parent.)

2. **same_regulated_domain**: Both regulate the same kind of activity or entity in their respective jurisdictions.
   (PASS if both deal with the same regulatory subject area. FAIL if different subject areas.)

3. **same_normative_role**: Both play the same structural role in normative statements.
   (PASS if both are norm subjects, OR both are norm objects, OR both are conditions, etc.)

4. **compatible_scope**: Both apply at the same scope level.
   (PASS if both individual-level, OR both organisational-level, OR both procedural-level.)

5. **swap_test**: A regulation written about one class could be sensibly rewritten substituting the other class, preserving the regulatory STRUCTURE (e.g., "obligation-on-party-X-to-do-Y" pattern).
   - PASS if substitution maintains the obligation/permission/prohibition pattern AND the regulated parties are equivalent types, even if the specific action differs.
   - FAIL ONLY if substitution changes the normative type or entity types.
   - Do NOT FAIL just because specific actions differ — focus on STRUCTURAL substitutability.

ALIGNMENT RELATION CHOICE (only if all 5 criteria PASS):

Choose ONE alignment_relation from:

- **owl:equivalentClass**: ONLY if classes are legally AND functionally identical (same statute, same regulatory body, same legal effect). EXTREMELY RARE for cross-jurisdiction alignment.

- **skos:closeMatch**: DEFAULT for cross-jurisdiction alignment. Use when classes are functionally similar but exist in different legal frameworks (UK vs US vs CA vs AU laws). Almost all cross-jurisdiction alignments should use this.

- **skos:broadMatch**: Class A is BROADER than Class B (A conceptually subsumes B).

- **skos:narrowMatch**: Class A is NARROWER than Class B (B conceptually subsumes A).

DECISION RULE:
- decision = "ALIGN" only if ALL FIVE criteria PASS
- decision = "REJECT" if ANY criterion fails
- If REJECT, set alignment_relation to null
- If ALIGN, choose appropriate alignment_relation (default: skos:closeMatch)

Return this exact JSON shape:
{{
  "decision": "ALIGN|REJECT",
  "alignment_relation": "owl:equivalentClass|skos:closeMatch|skos:broadMatch|skos:narrowMatch|null",
  "criteria": {{
    "same_cco_parent":     {{ "result": "PASS|FAIL", "justification": "1 sentence" }},
    "same_regulated_domain": {{ "result": "PASS|FAIL", "justification": "1 sentence" }},
    "same_normative_role": {{ "result": "PASS|FAIL", "justification": "1 sentence" }},
    "compatible_scope":    {{ "result": "PASS|FAIL", "justification": "1 sentence" }},
    "swap_test":           {{ "result": "PASS|FAIL", "justification": "1 sentence" }}
  }},
  "overall_justification": "2-3 sentence summary of the decision"
}}"""

print(' Prompt template defined (with alignment_relation choice)')

In [ ]:
# -- Step 2 LLM call helper --------------------------------------------------
client = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)

def _extract_first_json_object(text):
    """Extract first balanced JSON object from a text response."""
    start = text.find('{')
    if start == -1:
        raise ValueError('No JSON object found in response')
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if escape:
            escape = False
            continue
        if ch == '\\':
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    raise ValueError('Unbalanced JSON braces')

def llm_validate_candidate(candidate, retries=2):
    """Call LLM for one candidate pair; return parsed JSON or error dict."""
    a, b = candidate['class_a'], candidate['class_b']
    user_msg = LLM_USER_TEMPLATE.format(
        jur_a=a['jurisdiction'],   uri_a=a['class'],   parent_a=a['parent'],
        label_a=a['label'],        def_a=a['definition'],
        jur_b=b['jurisdiction'],   uri_b=b['class'],   parent_b=b['parent'],
        label_b=b['label'],        def_b=b['definition'],
        sim=candidate['similarity'],
    )
    
    last_err = None
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {'role': 'system', 'content': LLM_SYSTEM},
                    {'role': 'user',   'content': user_msg},
                ],
                temperature=LLM_TEMPERATURE,
                max_tokens=LLM_MAX_TOKENS,
            )
            raw = resp.choices[0].message.content
            json_str = _extract_first_json_object(raw)
            parsed = json.loads(json_str)
            return {'ok': True, 'parsed': parsed, 'raw': raw}
        except Exception as e:
            last_err = str(e)[:200]
            if attempt < retries:
                time.sleep(2 ** attempt)
                continue
    return {'ok': False, 'error': last_err}

print(' LLM helper defined')

In [ ]:
# -- Resume from checkpoint if exists ---------------------------------------
validations = []
raw_lines = []
n_align = 0
n_reject = 0
n_error = 0
start_idx = 0

if RESUME_FROM_CHECKPOINT:
    existing_checkpoints = sorted(CHECKPOINT_DIR.glob('checkpoint_*.json'))
    if existing_checkpoints:
        latest = existing_checkpoints[-1]
        print(f'Found existing checkpoint: {latest.name}')
        with open(latest, encoding='utf-8') as f:
            checkpoint = json.load(f)
        validations = checkpoint['validations']
        n_align = checkpoint['n_align']
        n_reject = checkpoint['n_reject']
        n_error = checkpoint['n_error']
        start_idx = checkpoint['last_idx']
        
        # Also restore raw responses
        if RAW_RESPONSES.exists():
            with open(RAW_RESPONSES, encoding='utf-8') as f:
                raw_lines = [json.loads(line) for line in f]
        
        print(f' Resuming from candidate {start_idx + 1}/{len(candidates)}')
        print(f'  Previous progress: ALIGN={n_align}, REJECT={n_reject}, ERROR={n_error}')
    else:
        print('No checkpoint found, starting fresh')
else:
    print('Resume disabled, starting fresh')

In [ ]:
# -- Step 2 run: validate every candidate with checkpoints ------------------
print(f'Validating {len(candidates) - start_idx} candidates (from {start_idx + 1}) with {LLM_MODEL}...')
print(f'Checkpoint will be saved every {CHECKPOINT_INTERVAL} candidates')
print('=' * 80)

for idx in range(start_idx, len(candidates)):
    cand = candidates[idx]
    a, b = cand['class_a'], cand['class_b']
    
    result = llm_validate_candidate(cand)
    
    record = {
        'candidate_idx':  idx + 1,
        'similarity':     cand['similarity'],
        'class_a':        a['class'],
        'class_b':        b['class'],
        'jurisdiction_a': a['jurisdiction'],
        'jurisdiction_b': b['jurisdiction'],
        'parent':         a['parent'],
    }
    
    if result['ok']:
        p = result['parsed']
        record['decision']              = p.get('decision', 'PARSE_ERROR')
        record['alignment_relation']    = p.get('alignment_relation', None)  # ← NEW
        record['criteria']              = p.get('criteria', {})
        record['overall_justification'] = p.get('overall_justification', '')
        if record['decision'] == 'ALIGN':
            n_align += 1
        elif record['decision'] == 'REJECT':
            n_reject += 1
        else:
            n_error += 1
    else:
        record['decision'] = 'LLM_ERROR'
        record['error']    = result['error']
        n_error += 1
    
    validations.append(record)
    raw_lines.append({
        'candidate_idx': idx + 1,
        'class_a':       a['class'],
        'class_b':       b['class'],
        'raw':           result.get('raw', ''),
        'error':         result.get('error', None),
    })
    
    # Progress print
    status = record['decision']
    status_marker = {'ALIGN': '[ALIGN]', 'REJECT': '[REJECT]', 'LLM_ERROR': '[ERROR]', 'PARSE_ERROR': '[ERROR]'}.get(status, '[?]')   
    print(f'[{idx+1:5d}/{len(candidates):5d}] [{cand["similarity"]:.3f}] {a["jurisdiction"]}/{a["class"].split(":")[-1][:30]:30s} ↔ {b["jurisdiction"]}/{b["class"].split(":")[-1][:30]:30s} {status_marker} {status}')    
    #  Save checkpoint every CHECKPOINT_INTERVAL candidates
    if (idx + 1) % CHECKPOINT_INTERVAL == 0:
        checkpoint_path = CHECKPOINT_DIR / f'checkpoint_{idx+1:05d}.json'
        checkpoint_data = {
            'last_idx':       idx + 1,
            'total_candidates': len(candidates),
            'validations':    validations,
            'n_align':        n_align,
            'n_reject':       n_reject,
            'n_error':        n_error,
            'progress_pct':   round((idx + 1) / len(candidates) * 100, 1),
            'saved_at':       datetime.now().isoformat(),
        }
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(checkpoint_data, f, indent=2, ensure_ascii=False)
        
        # Save raw responses incrementally
        with open(RAW_RESPONSES, 'w', encoding='utf-8') as f:
            for line in raw_lines:
                f.write(json.dumps(line, ensure_ascii=False) + '\n')
        
        print(f'   CHECKPOINT SAVED: {checkpoint_path.name}')
        print(f'     Progress: {idx+1}/{len(candidates)} ({(idx+1)/len(candidates)*100:.1f}%)')
        print(f'     ALIGN={n_align} | REJECT={n_reject} | ERROR={n_error}')
        print('  ' + '-' * 76)

# Final save
print('=' * 80)
print(f'\n FINAL TOTALS:')
print(f'  ALIGN:  {n_align}')
print(f'  REJECT: {n_reject}')
print(f'  ERROR:  {n_error}')

# Save final raw responses
with open(RAW_RESPONSES, 'w', encoding='utf-8') as f:
    for line in raw_lines:
        f.write(json.dumps(line, ensure_ascii=False) + '\n')
print(f'\n Raw responses saved: {RAW_RESPONSES}')

In [ ]:
# -- Step 3: Cluster aligned pairs + emit correct alignment axioms ----------
from rdflib import Namespace

SKOS_NS = Namespace('http://www.w3.org/2004/02/skos/core#')

aligned_pairs = [(v['class_a'], v['class_b'], v.get('alignment_relation', 'skos:closeMatch')) 
                 for v in validations if v['decision'] == 'ALIGN']
print(f'Aligned pairs to consolidate: {len(aligned_pairs)}')

# Distribution by relation
from collections import Counter
rel_dist = Counter(rel for _, _, rel in aligned_pairs)
print('\nAlignment relation distribution:')
for rel, n in rel_dist.most_common():
    print(f'  {rel}: {n}')

# Build index map
qname_to_idx = {c['class']: i for i, c in enumerate(all_classes)}

# Union-find ONLY for owl:equivalentClass (transitive closure makes sense)
# For skos relations, keep pairwise (no transitive clustering)
equiv_pairs = [(a, b) for a, b, rel in aligned_pairs if rel == 'owl:equivalentClass']
skos_pairs = [(a, b, rel) for a, b, rel in aligned_pairs if rel != 'owl:equivalentClass']

# Union-find for equivalentClass clusters
parent_uf = list(range(len(all_classes)))
def find(x):
    while parent_uf[x] != x:
        parent_uf[x] = parent_uf[parent_uf[x]]
        x = parent_uf[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent_uf[ra] = rb

for a_qname, b_qname in equiv_pairs:
    if a_qname in qname_to_idx and b_qname in qname_to_idx:
        union(qname_to_idx[a_qname], qname_to_idx[b_qname])

# Group equivalentClass clusters
clusters_raw = defaultdict(list)
for idx in range(len(all_classes)):
    clusters_raw[find(idx)].append(idx)
multi_clusters = [v for v in clusters_raw.values() if len(v) > 1]

print(f'\nowl:equivalentClass clusters: {len(multi_clusters)}')
for k, indices in enumerate(multi_clusters, 1):
    cls_list = [all_classes[i] for i in indices]
    jurs = sorted(set(c['jurisdiction'] for c in cls_list))
    print(f'  Cluster {k}: {len(indices)} classes across {len(jurs)} jurisdictions ({cls_list[0]["parent"]})')
    for c in cls_list:
        print(f'    {c["jurisdiction"]}: {c["class"]} — "{c["label"]}"')

print(f'\nskos pairwise alignments: {len(skos_pairs)}')
for a, b, rel in skos_pairs[:20]:  # Show first 20
    print(f'  {a} --{rel}--> {b}')
if len(skos_pairs) > 20:
    print(f'  ... and {len(skos_pairs) - 20} more')

# Build TTL with correct predicates
def qname_to_uri(qname):
    if ':' in qname:
        pfx, local = qname.split(':', 1)
        if pfx in JUR_NAMESPACES:
            return URIRef(JUR_NAMESPACES[pfx] + local)
    return URIRef(qname)

def relation_to_predicate(rel_str):
    """Map relation string to RDF predicate."""
    mapping = {
        'owl:equivalentClass': OWL.equivalentClass,
        'skos:closeMatch':     SKOS_NS.closeMatch,
        'skos:broadMatch':     SKOS_NS.broadMatch,
        'skos:narrowMatch':    SKOS_NS.narrowMatch,
    }
    return mapping.get(rel_str, SKOS_NS.closeMatch)  # Default

align_graph = Graph()
for pfx, uri in JUR_NAMESPACES.items():
    align_graph.bind(pfx, uri)
align_graph.bind('owl', OWL)
align_graph.bind('skos', SKOS_NS)

# Emit equivalentClass for transitive clusters
for indices in multi_clusters:
    uris = [qname_to_uri(all_classes[i]['class']) for i in indices]
    for i in range(len(uris)):
        for j in range(i+1, len(uris)):
            align_graph.add((uris[i], OWL.equivalentClass, uris[j]))

# Emit skos relations for pairwise alignments
for a_qname, b_qname, rel in skos_pairs:
    a_uri = qname_to_uri(a_qname)
    b_uri = qname_to_uri(b_qname)
    predicate = relation_to_predicate(rel)
    align_graph.add((a_uri, predicate, b_uri))

align_graph.serialize(destination=str(ALIGNMENT_TTL), format='turtle')
print(f'\n Alignment TTL saved: {ALIGNMENT_TTL} ({len(align_graph)} triples)')

# Verify
try:
    verify = Graph()
    verify.parse(str(ALIGNMENT_TTL), format='turtle')
    print(f' Parse verification: OK ({len(verify)} triples)')
except Exception as e:
    print(f' Parse verification FAILED: {e}')

In [ ]:
# -- Save full alignment manifest --------------------------------------------
cluster_records = []
for k, indices in enumerate(multi_clusters, 1):
    cls_list = [all_classes[i] for i in indices]
    cluster_records.append({
        'cluster_id': k,
        'size':       len(cls_list),
        'jurisdictions': sorted(set(c['jurisdiction'] for c in cls_list)),
        'parent':     cls_list[0]['parent'],
        'members': [
            {'jurisdiction': c['jurisdiction'], 'class': c['class'],
             'label': c['label'], 'definition': c['definition']}
            for c in cls_list
        ],
    })

manifest = {
    'metadata': {
        'run_type':              'stage5b_alignment_full_run',
        'stage':                 'stage5b_full',
        'embed_model':           EMBED_MODEL,
        'llm_model':             LLM_MODEL,
        'sim_threshold':         SIM_THRESHOLD,
        'same_parent_required':  SAME_PARENT_REQUIRED,
        'created_at':            datetime.now().isoformat(),
        'stage5a_source':        str(CANONICAL_CLASSES_PATH),
        'checkpoint_interval':   CHECKPOINT_INTERVAL,
    },
    'totals': {
            'n_canonical_input':        len(all_classes),
            'n_embedding_candidates':   len(candidates),
            'n_llm_align':              n_align,
            'n_llm_reject':             n_reject,
            'n_llm_error':              n_error,
            'n_clusters':               len(multi_clusters),
            'n_classes_aligned':        sum(c['size'] for c in cluster_records),
            'n_equivalentClass_axioms': len(align_graph),
            'alignment_relation_distribution': dict(rel_dist),  # ← NEW
        },
    'clusters':    cluster_records,
    'validations': validations,
}

with open(ALIGNMENT_MANIFEST, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f' Manifest saved: {ALIGNMENT_MANIFEST}')

In [ ]:
# -- Final summary -----------------------------------------------------------
print('=' * 70)
print('STAGE 5B FULL RUN SUMMARY')
print('=' * 70)
print(f'  Canonical classes input:              {len(all_classes)}')
print(f'  Embedding candidates (sim >= {SIM_THRESHOLD}):  {len(candidates)}')
print(f'  LLM ALIGN:                            {n_align}')
print(f'  LLM REJECT:                           {n_reject}')
print(f'  LLM ERROR:                            {n_error}')
print()

# Alignment relation distribution
print('Alignment relation distribution:')
for rel, n in rel_dist.most_common():
    pct = (n / n_align * 100) if n_align > 0 else 0
    print(f'  {rel}: {n} ({pct:.1f}%)')
print(f'  owl:equivalentClass: {rel_dist.get("owl:equivalentClass", 0)}')
print()

# Classes participating in alignments (corrected: includes skos pairs)
classes_in_alignments = set()
for v in validations:
    if v['decision'] == 'ALIGN':
        classes_in_alignments.add(v['class_a'])
        classes_in_alignments.add(v['class_b'])

print(f'  Total alignment axioms emitted:       {len(align_graph)}')
print(f'  Unique classes in alignments:         {len(classes_in_alignments)}')
print(f'  owl:equivalentClass clusters:         {len(multi_clusters)}')
print()

# Per-jurisdiction participation (corrected: counts all alignments)
jur_participation = Counter()
for v in validations:
    if v['decision'] == 'ALIGN':
        jur_participation[v['jurisdiction_a']] += 1
        jur_participation[v['jurisdiction_b']] += 1

print('Per-jurisdiction participation in alignments:')
for j in JURISDICTIONS:
    print(f'  {j}: {jur_participation.get(j, 0)} alignments')

# Per-jurisdiction pair distribution
pair_dist = Counter()
for v in validations:
    if v['decision'] == 'ALIGN':
        pair = tuple(sorted([v['jurisdiction_a'], v['jurisdiction_b']]))
        pair_dist[pair] += 1

print('\nAlignments by jurisdiction pair:')
for pair, n in pair_dist.most_common():
    print(f'  {pair[0]} <-> {pair[1]}: {n}')

# Per-parent distribution
parent_align = Counter()
for v in validations:
    if v['decision'] == 'ALIGN':
        parent_align[v['parent']] += 1

print('\nAlignments by CCO parent:')
for p, n in parent_align.most_common():
    print(f'  {p}: {n}')

# LLM filter rate
if len(candidates) > 0:
    filter_rate = n_align / len(candidates)
    print(f'\nLLM filter rate (align / candidates): {filter_rate:.1%}')

print(f'\nOutput files:')
print(f'  Alignment TTL:     {ALIGNMENT_TTL.name}')
print(f'  Manifest:          {ALIGNMENT_MANIFEST.name}')
print(f'  Raw LLM responses: {RAW_RESPONSES.name}')
print(f'  Checkpoints:       {CHECKPOINT_DIR}/')